# A role's mode is declared once, and a wrong one is refused where it is written.

Matching [script](17_a_wrong_role_mode.py). Run these declaration cases in order.


In [1]:
"""A role's mode is declared once, and a wrong one is refused where it is written."""

from __future__ import annotations

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.exceptions import RoleSpecTypeError
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.role_mode import RoleMode, role_mode



## A member of the enum, and nothing else

In [2]:
@role_mode(RoleMode.ALIVE)
class ViewerRole(ApplicationRole):
    """Read the catalog."""

    name = "viewer"
    description = "Can read the catalog"


print("declared:", ViewerRole._role_mode_info)


declared: {'mode': <RoleMode.ALIVE: 'alive'>}


## The string the enum carries is not the member

In [3]:
try:

    @role_mode("alive")
    class MistypedRole(ApplicationRole):
        """Written the way the value looks."""

        name = "mistyped"
        description = "Carries a string instead of a member"

except Exception as error:  # the declaration fails; the class name is never bound
    print(type(error).__name__ + ":", error)


RoleModeDeclarationError: @role_mode(MistypedRole): expected a RoleMode member, got 'alive'. Allowed: ALIVE, DEPRECATED, SILENCED, UNUSED.


## The parentheses are part of the call

In [4]:
try:

    @role_mode
    class ParenthesislessRole(ApplicationRole):
        """Forgot the call."""

        name = "parenthesisless"
        description = "Decorated without parentheses"

except Exception as error:
    print(type(error).__name__ + ":", error)


RoleModeDeclarationError: @role_mode: expected a RoleMode member, got the class ParenthesislessRole itself — the parentheses are missing. Write @role_mode(RoleMode.ALIVE). Allowed: ALIVE, DEPRECATED, SILENCED, UNUSED.


## A member decides nothing by itself; the consumers do

In [5]:
@role_mode(RoleMode.UNUSED)
class RetiredRole(ApplicationRole):
    """Kept in the code, never required."""

    name = "retired"
    description = "Kept, unused"


print("declared:", RetiredRole._role_mode_info)


try:

    @check_roles(RetiredRole)
    class WithdrawAction:
        """Would have required a role that is not used."""

except RoleSpecTypeError as error:
    print(type(error).__name__ + ":", error)


declared: {'mode': <RoleMode.UNUSED: 'unused'>}
RoleSpecTypeError: @check_roles cannot require role 'RetiredRole': it is marked RoleMode.UNUSED.
